In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import strawberryfields as sf
from strawberryfields.ops import*
from numpy.polynomial.hermite import hermgauss
from scipy.special import eval_hermite, factorial

In [ ]:
def sf_protocol(eta, var, key_len):


    bob_bitstr = []
    alice_bitstr = []
    for i in range(key_len):
        prog = sf.Program(2)
        theta = np.arccos(np.sqrt(eta))
        with prog.context as q:

            # Input state on Alice's mode
            Vac | q[0]
            alpha_p = np.random.normal(0, np.sqrt(var))
            alpha_q = np.random.normal(0, np.sqrt(var))
            Xgate(alpha_q) | q[0]
            Zgate(alpha_p) | q[0]

            # Vacuum entering unused beam-splitter port
            Vac | q[1]
            # Beam splitter
            BSgate(theta, 0.0) | (q[0], q[1])

            x = np.random.randint(0, 2)
            if x==0:
                MeasureHomodyne(0) | q[0]
                alice_bitstr.append(alpha_q)
                
            else:
                MeasureHomodyne(np.pi/2) | q[0]
                alice_bitstr.append(alpha_p)


        eng = sf.Engine("gaussian")
        result = eng.run(prog)
        bob_bitstr.append(result.samples[0][0])
        eng.reset()

    return alice_bitstr, bob_bitstr



In [ ]:
eta = 0.3
keylen=1000
sigma_grid = np.linspace(0,100,101)
I_grid=[]
for sigma in sigma_grid:
    alice_bitstr, bob_bitstr = (eta, sigma**2, keylen)
    rho = np.corrcoef(alice_bitstr, bob_bitstr)[0, 1]
    I_AB = -0.5 * np.log2(1 - rho**2)
    I_grid.append(I_AB)

In [ ]:
plt.plot(sigma_grid, I_grid)
plt.plot(sigma_grid, 1/2*np.log2(1+(sigma_grid**2*eta)/(1+eta*(1-1))))